# 🏥 ETL FarmaVida — Resolução Completa

**Disciplina:** Engenharia de Dados · **Prof. Lucas Vinicius Oliveira Cardoso**

Pipeline ETL que integra **3 sistemas operacionais independentes** (Vendas, Estoque, RH)
em um **Data Warehouse estrela** no SQLite, seguido de validação e 8 consultas analíticas.

### Inconsistências propositais nos dados de origem
| Aspecto | Vendas (PDV) | Estoque | RH |
|---|---|---|---|
| **Datas** | `DD/MM/YYYY` (texto) | `YYYY-MM-DD` (DATE) | `MM-DD-YYYY` (texto) |
| **Valores** | Centavos (integer) | Decimal `NUMERIC(10,2)` | Decimal |
| **Filiais** | `codigo_filial` (1–7) | `sigla_filial` (FL-UF-NN) | `id_filial_rh` (1–7) |
| **Status** | `A` / `C` | `ativo` / `inativo` | `1` / `0` |
| **CPF** | Com pontuação | — | Sem pontuação (`nr_cpf`) |
| **Produtos** | Nome livre (case variado) | `descricao` UPPER oficial | — |

## 1. Dependências

In [ ]:
%pip install psycopg2-binary pandas matplotlib seaborn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 60.9 MB/s eta 0:00:00


In [ ]:
import psycopg2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import os
import re
from datetime import datetime

# Configurações de exibição
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

print("✅ Bibliotecas carregadas")

✅ Bibliotecas carregadas


---
## 2. Conexão com os 3 Bancos Neon

Cada sistema operacional vive em um **projeto Neon separado**.

In [ ]:
# ======================== CONFIGURAÇÃO ========================

CREDENCIAIS = dict(user="aluno_readonly", password="Alunos2026!", dbname="neondb", sslmode="require")

HOSTS = {
    "vendas":  "ep-raspy-block-ainzo20a-pooler.c-4.us-east-1.aws.neon.tech",
    "estoque": "ep-dry-frog-ail6dwj9-pooler.c-4.us-east-1.aws.neon.tech",
    "rh":      "ep-noisy-unit-aiyj66lx-pooler.c-4.us-east-1.aws.neon.tech",
}

def extrair_tabelas(host, tabelas):
    """Conecta no Neon e retorna um dict {nome_tabela: DataFrame}."""
    conn = psycopg2.connect(host=host, **CREDENCIAIS)
    resultado = {}
    for tabela in tabelas:
        resultado[tabela] = pd.read_sql(f"SELECT * FROM {tabela}", conn)
        print(f"   📥 {tabela}: {len(resultado[tabela])} linhas, {len(resultado[tabela].columns)} colunas")
    conn.close()
    return resultado

# Teste rápido de conectividade
for nome, host in HOSTS.items():
    try:
        c = psycopg2.connect(host=host, **CREDENCIAIS)
        c.close()
        print(f"✅ {nome}: conectado")
    except Exception as e:
        print(f"❌ {nome}: {e}")

✅ vendas: conectado
✅ estoque: conectado
✅ rh: conectado


---
## 3. Extração (Extract)

### 3.1 Sistema de Vendas (PDV)
Tabelas: `filiais`, `formas_pagamento`, `vendas`, `itens_venda`

In [ ]:
print("🔵 Extraindo VENDAS...")
dados_vendas = extrair_tabelas(HOSTS["vendas"], ["filiais", "formas_pagamento", "vendas", "itens_venda"])

vd_filiais = dados_vendas["filiais"]
vd_formas  = dados_vendas["formas_pagamento"]
vd_vendas  = dados_vendas["vendas"]
vd_itens   = dados_vendas["itens_venda"]

print(f"\n📋 Colunas vendas:      {vd_vendas.columns.tolist()}")
print(f"📋 Colunas itens_venda: {vd_itens.columns.tolist()}")
print(f"📋 Colunas formas_pgto: {vd_formas.columns.tolist()}")
display(vd_vendas.head(3))

🔵 Extraindo VENDAS...


/tmp/ipython-input-1211332383.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  resultado[tabela] = pd.read_sql(f"SELECT * FROM {tabela}", conn)


   📥 filiais: 7 linhas, 4 colunas
   📥 formas_pagamento: 5 linhas, 3 colunas
   📥 vendas: 60 linhas, 10 colunas
   📥 itens_venda: 146 linhas, 7 colunas

📋 Colunas vendas:      ['id_venda', 'codigo_filial', 'id_vendedor', 'cpf_cliente', 'data_venda', 'hora_venda', 'id_forma_pgto', 'valor_total_centavos', 'desconto_centavos', 'status']
📋 Colunas itens_venda: ['id_item', 'id_venda', 'cod_produto', 'nome_produto', 'quantidade', 'preco_unitario_centavos', 'subtotal_centavos']
📋 Colunas formas_pgto: ['id_forma', 'descricao', 'permite_parcelamento']


,id_venda,codigo_filial,id_vendedor,cpf_cliente,data_venda,hora_venda,id_forma_pgto,valor_total_centavos,desconto_centavos,status
0,1,1,1001,123.456.789-00,01/10/2025,08:15:00,3,15490,0,A
1,2,1,1001,None,01/10/2025,08:42:00,1,2350,0,A
2,3,2,1005,987.654.321-00,01/10/2025,09:10:00,4,8920,500,A


### 3.2 Sistema de Estoque
Tabelas: `categorias`, `produtos`, `filiais_estoque`, `fornecedores`, `estoque`, `entradas_mercadoria`, `itens_entrada`

In [ ]:
print("🟢 Extraindo ESTOQUE...")
dados_estoque = extrair_tabelas(HOSTS["estoque"], [
    "categorias", "produtos", "filiais_estoque", "fornecedores",
    "estoque", "entradas_mercadoria", "itens_entrada"
])

es_categorias = dados_estoque["categorias"]
es_produtos   = dados_estoque["produtos"]
es_filiais    = dados_estoque["filiais_estoque"]
es_fornecedores = dados_estoque["fornecedores"]
es_estoque    = dados_estoque["estoque"]
es_entradas   = dados_estoque["entradas_mercadoria"]
es_itens_ent  = dados_estoque["itens_entrada"]

print(f"\n📋 Colunas produtos:    {es_produtos.columns.tolist()}")
print(f"📋 Colunas estoque:     {es_estoque.columns.tolist()}")
print(f"📋 Colunas fornecedores:{es_fornecedores.columns.tolist()}")
print(f"📋 Colunas entradas:    {es_entradas.columns.tolist()}")
print(f"📋 Colunas itens_ent:   {es_itens_ent.columns.tolist()}")
display(es_produtos.head(3))

🟢 Extraindo ESTOQUE...


/tmp/ipython-input-1211332383.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  resultado[tabela] = pd.read_sql(f"SELECT * FROM {tabela}", conn)


   📥 categorias: 6 linhas, 3 colunas
   📥 produtos: 13 linhas, 12 colunas
   📥 filiais_estoque: 7 linhas, 7 colunas
   📥 fornecedores: 7 linhas, 9 colunas
   📥 estoque: 60 linhas, 10 colunas
   📥 entradas_mercadoria: 43 linhas, 7 colunas
   📥 itens_entrada: 31 linhas, 6 colunas

📋 Colunas produtos:    ['id_produto', 'ean', 'descricao', 'id_categoria', 'unidade', 'preco_custo', 'preco_venda', 'margem_lucro', 'peso_gramas', 'controlado', 'ativo', 'data_cadastro']
📋 Colunas estoque:     ['id_estoque', 'id_produto', 'sigla_filial', 'quantidade', 'estoque_minimo', 'estoque_maximo', 'ultima_entrada', 'ultima_saida', 'lote', 'validade']
📋 Colunas fornecedores:['id_fornecedor', 'razao_social', 'nome_fantasia', 'cnpj_fornecedor', 'email', 'telefone', 'cidade', 'uf', 'ativo']
📋 Colunas entradas:    ['id_entrada', 'sigla_filial', 'id_fornecedor', 'numero_nf', 'data_entrada', 'valor_total', 'status']
📋 Colunas itens_ent:   ['id_item_entrada', 'id_entrada', 'id_produto', 'quantidade', 'preco_unitar

,id_produto,ean,descricao,id_categoria,unidade,preco_custo,preco_venda,margem_lucro,peso_gramas,controlado,ativo,data_cadastro
0,MED-0003,7891234560031,PARACETAMOL 750MG CX C/ 10 COMP,1,CX,8.50,23.50,176.47,45,False,ativo,2020-03-15
1,MED-0012,7891234560122,DIPIRONA SÓDICA 500MG CX 20 COMP,1,CX,32.00,84.90,165.31,80,False,ativo,2020-03-15
2,MED-0101,7891234561012,LOSARTANA POTÁSSICA 50MG CX 30,1,CX,45.00,125.00,177.78,110,True,ativo,2020-06-01


### 3.3 Sistema de RH
Tabelas: `cargos`, `departamentos`, `filiais_rh`, `funcionarios`, `escalas`, `historico_salario`

In [ ]:
print("🟠 Extraindo RH...")
dados_rh = extrair_tabelas(HOSTS["rh"], [
    "cargos", "departamentos", "filiais_rh", "funcionarios",
    "escalas", "historico_salario"
])

rh_cargos   = dados_rh["cargos"]
rh_deptos   = dados_rh["departamentos"]
rh_filiais  = dados_rh["filiais_rh"]
rh_funcs    = dados_rh["funcionarios"]
rh_escalas  = dados_rh["escalas"]
rh_hist_sal = dados_rh["historico_salario"]

print(f"\n📋 Colunas funcionarios: {rh_funcs.columns.tolist()}")
print(f"📋 Colunas filiais_rh:   {rh_filiais.columns.tolist()}")
display(rh_funcs[["matricula", "nome_completo", "nr_cpf", "data_nascimento", "data_admissao", "status"]].head(5))

🟠 Extraindo RH...


/tmp/ipython-input-1211332383.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  resultado[tabela] = pd.read_sql(f"SELECT * FROM {tabela}", conn)


   📥 cargos: 10 linhas, 5 colunas
   📥 departamentos: 5 linhas, 3 colunas
   📥 filiais_rh: 7 linhas, 7 colunas
   📥 funcionarios: 24 linhas, 15 colunas
   📥 escalas: 44 linhas, 6 colunas
   📥 historico_salario: 17 linhas, 6 colunas

📋 Colunas funcionarios: ['matricula', 'nome_completo', 'nr_cpf', 'data_nascimento', 'data_admissao', 'data_demissao', 'id_cargo', 'id_departamento', 'id_filial_rh', 'salario', 'email', 'telefone', 'endereco', 'cep', 'status']
📋 Colunas filiais_rh:   ['id_filial_rh', 'nome_filial', 'regiao', 'cidade_rh', 'uf_rh', 'gerente_responsavel', 'status']


,matricula,nome_completo,nr_cpf,data_nascimento,data_admissao,status
0,1001,Maria Fernanda Oliveira Silva,12345678900,03-15-1992,02-10-2020,1
1,1002,Carlos Eduardo Santos,98765432100,07-22-1988,05-15-2019,1
2,1003,Ana Carolina Mendes,11122233344,11-08-1985,01-10-2018,1
3,1004,Lucas Henrique Pereira,55566677788,01-30-1995,08-01-2021,1
4,1005,Juliana Cristina Almeida,22233344455,05-18-1990,03-20-2020,1


In [ ]:
# ── Fechar conexão do DW ──────────────────────────────────────
try:
    conn_dw.close()
    print("✅ Conexão DW (SQLite) fechada.")
except Exception:
    print("⚠️  Conexão DW já estava fechada.")

print(f"\n📁 Arquivo gerado: {DW_PATH}")
print(f"   Tamanho: {os.path.getsize(DW_PATH) / 1024:.1f} KB")
print("\n🎉 Pipeline ETL concluído com sucesso!")

✅ Conexão DW (SQLite) fechada.

📁 Arquivo gerado: /content/farmavida_dw.sqlite
   Tamanho: 208.0 KB

🎉 Pipeline ETL concluído com sucesso!


---

### ✅ Resumo do Pipeline

| Etapa | Descrição |
|-------|-----------|
| **Extract** | Dados extraídos de 3 bancos PostgreSQL (Neon) — Vendas, Estoque e RH |
| **Diagnóstico** | Identificadas inconsistências: datas em texto, valores em centavos, CPF sem máscara, status heterogêneo, filiais com chaves diferentes |
| **Transform** | Padronização de datas, conversão de valores, limpeza de CPF, normalização de status, mapeamento unificado de filiais |
| **Modelagem** | Data Warehouse em SQLite com esquema estrela: 5 dimensões + 3 fatos |
| **Load** | Carga completa com verificação de contagens |
| **Análise** | 8 consultas analíticas com visualizações gráficas |

> **Arquivo gerado:** `farmavida_dw.db` (SQLite) — pode ser aberto com DB Browser, DBeaver ou qualquer cliente SQLite.

---
*Resolução elaborada pelo Prof. Lucas Vinicius Oliveira Cardoso — Engenharia de Dados, 2025.*